# 🎙️ Vonigen by Aniki — Engine Notebook

Notebook ini adalah **mesin** voice cloning Anda. Aplikasi web **Vonigen by Aniki** hanya tampilan depan;
suara asli dibuat di sini memakai GPU gratis Google Colab.

Engine: **OmniVoice** (`k2-fsa/OmniVoice`) — voice cloning zero-shot dengan dukungan
**bahasa Indonesia**, cukup 3–10 detik audio referensi.

## 📋 Petunjuk Penggunaan

1. Klik menu **Runtime > Change runtime type >** pilih **T4 GPU** > **Save**.
2. Isi dua nilai di sel **⚙️ Pengaturan** di bawah (alamat aplikasi Vonigen by Aniki + Pairing Secret).
3. Jalankan proses:
   - **PC / Laptop:** klik tulisan **Run All** di atas.
   - **HP / Smartphone:** klik garis tiga di kiri atas > pilih **Runtime** > **Run All**.
4. Tunggu 3–6 menit (install + unduh model). Akan muncul banner **KODE AKSES** berisi 6 digit.
5. Salin 6 digit tersebut ke halaman login aplikasi Vonigen by Aniki.

> ⚠️ **Penting:** Tetap buka halaman Colab ini selama dipakai. Kalau tab/runtime Colab dimatikan,
> kode akses mati dan aplikasi tidak bisa generate lagi sampai notebook dijalankan ulang.


## ⚙️ 1. Pengaturan

Ganti dua nilai di bawah, lalu lanjutkan ke sel berikutnya.
- **APP_URL** = alamat aplikasi Vonigen by Aniki (lihat di address bar browser, harus lengkap sampai nama aplikasi).
- **PAIRING_SECRET** = kunci dari halaman **Admin > Pairing Secret** di aplikasi Anda.

In [ ]:
# ============================ PENGATURAN ============================
APP_URL = ""   # <-- masukkan alamat aplikasi Anda (misal https://...)
PAIRING_SECRET = "DzwNNj0TMpfc0BzIWhZzBCtl6Y0"          # <-- ganti (ambil dari halaman Admin/Pengaturan)

# Opsional
FIXED_CODE = ""            # paksa kode akses 6 digit (kosong = acak). Contoh: "123456"
PORT = 7860                # port internal Gradio di Colab (biarkan)
DEFAULT_LANGUAGE = "Indonesian"
# ===================================================================

## 📦 2. Install dependensi (sekali jalan, ~2-4 menit)

> ℹ️ Sel ini memasang `omnivoice` (butuh `torch`, `torchaudio`, `transformers`). Kalau setelah sel ini
> muncul error impor/mismatch (misalnya `ImportError` atau pesan soal numpy/torch), lakukan
> **Runtime > Restart session**, lalu **Run All** lagi — pemasangan tadi sudah tersimpan di cache Colab,
> jadi proses kedua jauh lebih cepat.

In [ ]:
import io, json, math, os, random, re, shutil, subprocess, sys, threading, time, uuid, warnings
from contextlib import redirect_stdout, redirect_stderr

warnings.filterwarnings("ignore")

def pip(*pkgs):
    print(">>> pip install", " ".join(pkgs), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

pip("omnivoice", "pycloudflared", "requests", "soundfile", "num2words", "accelerate")

# Pastikan paket engine benar-benar bisa dimuat — pesan errornya dibuat jelas kalau gagal.
try:
    import omnivoice  # noqa: F401
    print("Engine omnivoice siap dimuat.")
except Exception as e:
    print("\n\033[1;31m[GAGAL MEMUAT ENGINE]\033[0m", type(e).__name__, e)
    print("Langkah yang dianjurkan: Runtime > Restart session, lalu Run All lagi.")
    print("(Pemasangan paket tadi sudah tersimpan, jadi proses ulang jauh lebih cepat.)\n")
    raise

import numpy as np
import torch
import soundfile as sf
import requests
import gradio as gr
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

DEVICE = get_best_device()
GPU_NAME = torch.cuda.get_device_name(0) if str(DEVICE).startswith("cuda") else "CPU"
print("Gradio:", gr.__version__)
print("Torch :", torch.__version__)
print("Device:", DEVICE, "|", GPU_NAME)
if DEVICE == "cpu":
    print("\n\033[1;31m[PERINGATAN] Runtime masih CPU! Ubah: Runtime > Change runtime type > T4 GPU, lalu Run All lagi.\033[0m\n")

## 🧠 3. Muat model OmniVoice

In [ ]:
MODEL_ID = "k2-fsa/OmniVoice"
ENGINE_USED = ""

_quiet = io.StringIO()
with redirect_stdout(_quiet), redirect_stderr(_quiet):
    MODEL = OmniVoice.from_pretrained(
        MODEL_ID,
        device_map=DEVICE,
        dtype=torch.float16 if str(DEVICE).startswith("cuda") else torch.float32,
        load_asr=True,
    )
ENGINE_USED = "OmniVoice (k2-fsa)"
SAMPLE_RATE = int(MODEL.sampling_rate)
print("Engine aktif:", ENGINE_USED, "| sample rate:", SAMPLE_RATE)


## 🎛️ 4. Fungsi inti

Dua fungsi yang dipanggil aplikasi web:
- `generate_voice(text, ref_audio, options_json)` → voice clone dari audio referensi
- `voice_design(text, instruct, options_json)` → racik suara dari deskripsi (tanpa referensi)


In [ ]:
OUT_DIR = "/content/voicegen_out"
os.makedirs(OUT_DIR, exist_ok=True)

LAST_ERROR = ""
JOB_COUNT = 0
FAIL_PREFIX = "GAGAL: "

def _set_error(msg):
    global LAST_ERROR
    LAST_ERROR = str(msg)[:300]
    print("[ERROR]", LAST_ERROR, flush=True)

def _clear_error():
    global LAST_ERROR
    LAST_ERROR = ""

def _ff(*args):
    if shutil.which("ffmpeg") is None:
        return False
    return subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", *args]).returncode == 0

def _as_path(ref):
    if isinstance(ref, dict):
        return ref.get("path") or ref.get("name") or ""
    return str(ref or "")

def _id_numbers(text):
    """Ubah angka & nominal rupiah menjadi kata (khusus bahasa Indonesia)."""
    try:
        from num2words import num2words
    except Exception:
        return text

    def rp(m):
        try:
            v = int(m.group(1).replace(".", ""))
            words = num2words(v, lang="id")
        except Exception:
            return m.group(0)
        return f"{words} rupiah"

    text = re.sub(r"\bRp\.?\s?(\d{1,3}(?:\.\d{3})+|\d+)", rp, text, flags=re.IGNORECASE)

    def num(m):
        try:
            return num2words(int(m.group(0)), lang="id")
        except Exception:
            return m.group(0)
        return f"{words}" if 'words' in locals() else re.sub(r"\b\d+\b", num, text)
    return re.sub(r"\b\d+\b", num, text)

def _parse_opts(options_json):
    try:
        opts = json.loads(options_json or "{}")
        if not isinstance(opts, dict):
            opts = {}
    except Exception:
        opts = {}
    return opts

def _lang(opts):
    lang = str(opts.get("language") or "").strip()
    return lang if lang and lang.lower() not in ("auto", "otomatis", "none") else DEFAULT_LANGUAGE

def _config(opts):
    return OmniVoiceGenerationConfig(
        num_step=int(opts.get("num_step", 32) or 32),
        guidance_scale=float(opts.get("guidance_scale", 2.0) or 2.0),
        denoise=bool(opts.get("denoise", True)),
        preprocess_prompt=bool(opts.get("preprocess_prompt", True)),
        postprocess_output=bool(opts.get("postprocess_output", True)),
    )

def _finish(wav, opts, tag, label):
    raw = os.path.join(OUT_DIR, f"raw_{tag}.wav")
    sf.write(raw, np.asarray(wav, dtype=np.float32), SAMPLE_RATE)
    speed = float(opts.get("speed", 1.0) or 1.0)
    norm = bool(opts.get("normalize", True))
    fmt = str(opts.get("format", "wav") or "wav").lower()
    chain = []
    if abs(speed - 1.0) > 0.01:
        chain.append(f"atempo={max(0.5, min(2.0, speed)):.4f}")
    if norm:
        chain.append("loudnorm=I=-16:TP=-1.5:LRA=11")
    out = os.path.join(OUT_DIR, f"out_{tag}.{'mp3' if fmt == 'mp3' else 'wav'}")
    args = ["-i", raw]
    if chain:
        args += ["-af", ",".join(chain)]
    if fmt == "mp3":
        args += ["-codec:a", "libmp3lame", "-b:a", "192k"]
    args += [out]
    if not _ff(*args):
        shutil.copyfile(raw, out)
    try:
        os.remove(raw)
    except Exception:
        pass
    dur = 0.0
    try:
        info = sf.info(out)
        dur = round(info.frames / info.samplerate, 1)
    except Exception:
        pass
    size_kb = round(os.path.getsize(out) / 1024, 1)
    text = f"OK · {label} · {dur}s audio · {size_kb} KB · {ENGINE_USED}"
    return out, text, dur

def generate_voice(text, ref_audio, options_json="{}", progress=None):
    global JOB_COUNT
    started = time.time()
    text = (text or "").strip()
    if not text:
        raise gr.Error("Naskah teks masih kosong.")
    opts = _parse_opts(options_json)
    path = _as_path(ref_audio)
    if not path or not os.path.exists(path):
        raise gr.Error("Audio referensi tidak ditemukan. Unggah atau rekam ulang (3-10 detik).")
    if bool(opts.get("normalize_numbers", True)) and "id" in _lang(opts).lower():
        text = _id_numbers(text)
    if progress:
        progress(0.15, desc="Menyiapkan prompt suara…")
    tag = uuid.uuid4().hex[:8]
    try:
        prompt = MODEL.create_voice_clone_prompt(
            ref_audio=path,
            ref_text=(str(opts.get("ref_text") or "").strip() or None),
            preprocess_prompt=bool(opts.get("preprocess_prompt", True)),
        )
        if progress:
            progress(0.45, desc="Mensintesis suara…")
        kw = dict(
            text=text,
            language=_lang(opts),
            voice_clone_prompt=prompt,
            generation_config=_config(opts),
            normalize_text=bool(opts.get("normalize_text", True)),
        )
        speed = float(opts.get("speed", 1.0) or 1.0)
        if abs(speed - 1.0) > 0.01:
            kw["speed"] = speed
        out_dur = float(opts.get("duration", 0) or 0)
        if out_dur > 0:
            kw["duration"] = out_dur
        audio = MODEL.generate(**kw)
    except Exception as e:
        _set_error(e)
        raise gr.Error(f"Gagal memproses suara: {e}")
    if progress:
        progress(0.85, desc="Menyimpan hasil…")
    file_path, info, dur = _finish(audio[0], opts, tag, "clone")
    _clear_error()
    JOB_COUNT += 1
    print(f"{info} · {round(time.time() - started, 1)}s proses", flush=True)
    return file_path, info

def voice_design(text, instruct, options_json="{}", progress=None):
    global JOB_COUNT
    text = (text or "").strip()
    instruct = (instruct or "").strip()
    if not text:
        raise gr.Error("Naskah teks masih kosong.")
    if not instruct:
        raise gr.Error("Pilih minimal satu karakter suara.")
    opts = _parse_opts(options_json)
    if bool(opts.get("normalize_numbers", True)) and "id" in _lang(opts).lower():
        text = _id_numbers(text)
    if progress:
        progress(0.3, desc="Meracik suara…")
    tag = uuid.uuid4().hex[:8]
    try:
        kw = dict(
            text=text,
            language=_lang(opts),
            instruct=instruct,
            generation_config=_config(opts),
            normalize_text=bool(opts.get("normalize_text", True)),
        )
        speed = float(opts.get("speed", 1.0) or 1.0)
        if abs(speed - 1.0) > 0.01:
            kw["speed"] = speed
        out_dur = float(opts.get("duration", 0) or 0)
        if out_dur > 0:
            kw["duration"] = out_dur
        audio = MODEL.generate(**kw)
    except Exception as e:
        _set_error(e)
        raise gr.Error(f"Gagal meracik suara: {e}")
    file_path, info, dur = _finish(audio[0], opts, tag, "design")
    _clear_error()
    JOB_COUNT += 1
    return file_path, info

print("Fungsi siap: generate_voice() dan voice_design()")

## 🚀 5. Bangun antarmuka Gradio (dipakai aplikasi web)

In [ ]:
INSTRUCT_MAP = {
    "Pria": "male", "Wanita": "female",
    "Anak-anak": "child", "Remaja": "teenager", "Dewasa": "young adult",
    "Paruh Baya": "middle-aged", "Lansia": "elderly",
    "Sangat Rendah": "very low pitch", "Rendah": "low pitch", "Sedang": "moderate pitch",
    "Tinggi": "high pitch", "Sangat Tinggi": "very high pitch",
    "Berbisik": "whisper",
    "Ceria": "cheerful", "Tenang": "calm tone", "Tegas": "confident tone",
    "Narator": "professional narrator", "Iklan": "energetic commercial voice",
    "Amerika": "american accent", "Inggris (UK)": "british accent",
    "Australia": "australian accent", "India": "indian accent",
}

CATEGORIES = {
    "Gender": ["Otomatis", "Pria", "Wanita"],
    "Usia": ["Otomatis", "Anak-anak", "Remaja", "Dewasa", "Paruh Baya", "Lansia"],
    "Pitch": ["Otomatis", "Sangat Rendah", "Rendah", "Sedang", "Tinggi", "Sangat Tinggi"],
    "Gaya": ["Otomatis", "Berbisik", "Ceria", "Tenang", "Tegas", "Narator", "Iklan"],
    "Aksen": ["Otomatis", "Amerika", "Inggris (UK)", "Australia", "India"],
}

LANGUAGES = ["Indonesian", "English", "Malay", "Arabic", "Hindi", "Japanese", "Korean",
             "Mandarin", "Spanish", "French", "German", "Portuguese", "Russian",
             "Turkish", "Dutch", "Italian", "Vietnamese", "Thai"]

THEME_CSS = """
footer { display: none !important; }
.gradio-container { max-width: 1100px !important; background: #f4f5f7 !important; }
:root, .dark, body, .gradio-container {
    --color-accent: #63d88c !important;
    --color-accent-soft-background-fill: #eafaf0 !important;
    --button-primary-background-fill: #111111 !important;
    --button-primary-text-color: #ffffff !important;
    --block-background-fill: #ffffff !important;
    --border-color-primary: #e5e7eb !important;
    --body-text-color: #111111 !important;
    --block-label-text-color: #111111 !important;
    --radius-lg: 16px !important;
    color: #111111 !important;
    background-color: #f4f5f7 !important;
}
.block, .form, .panel, fieldset {
    border-radius: 14px !important;
    border: 1px solid #e5e7eb !important;
    background: #ffffff !important;
    box-shadow: 0 1px 2px rgba(17,24,39,.05), 0 8px 24px rgba(17,24,39,.06) !important;
}
button.primary { font-weight: 700 !important; border-radius: 10px !important; }
button.primary:hover { background-color: #262626 !important; color: #63d88c !important; }
"""

def _instruct_from(groups):
    picked = [INSTRUCT_MAP[g] for g in groups if g and g != "Otomatis"]
    return ", ".join(picked)

def build_demo():
    with gr.Blocks(title="Vonigen by Aniki", css=THEME_CSS) as demo:
        gr.Markdown("## 🎙️ Vonigen by Aniki\nMesin voice cloning Anda. **Jangan tutup halaman ini** selama dipakai.")
        with gr.Tabs():
            with gr.Tab("Voice Clone"):
                with gr.Row():
                    with gr.Column():
                        vc_text = gr.Textbox(label="Naskah Teks", lines=6, placeholder="Ketik naskah di sini...")
                        vc_ref = gr.Audio(label="Audio Referensi (3-10 Detik)", type="filepath")
                        with gr.Accordion("Opsi Suara (Opsional)", open=False):
                            vc_lang = gr.Dropdown(LANGUAGES, value=DEFAULT_LANGUAGE, label="Bahasa")
                            vc_ref_text = gr.Textbox(label="Transkrip Audio Referensi (opsional)", lines=2,
                                                     placeholder="Isi kalau Anda tahu teks yang diucapkan di audio referensi")
                        with gr.Accordion("Pengaturan Lanjutan", open=False):
                            vc_steps = gr.Slider(8, 64, value=32, step=1, label="Inference Steps")
                            vc_cfg = gr.Slider(0.0, 4.0, value=2.0, step=0.1, label="Guidance Scale (CFG)")
                            vc_speed = gr.Slider(0.5, 1.5, value=1.0, step=0.05, label="Kecepatan")
                            vc_dur = gr.Number(label="Durasi Detik (0 = otomatis)", value=0)
                            vc_denoise = gr.Checkbox(label="Denoise Audio", value=True)
                            vc_pprompt = gr.Checkbox(label="Preprocess Prompt", value=True)
                            vc_pout = gr.Checkbox(label="Postprocess Hening", value=True)
                            vc_fmt = gr.Dropdown(["wav", "mp3"], value="wav", label="Format")
                            vc_norm = gr.Checkbox(label="Normalisasi Angka", value=True)
                    with gr.Column():
                        vc_out = gr.Audio(label="Hasil Audio", type="filepath")
                        vc_status = gr.Textbox(label="Status", value="🟢 Siap memproses suara.", lines=3)
                        vc_btn = gr.Button("Generate Voice Clone", variant="primary")

                def _vc_fn(text, ref, lang, ref_text, steps, cfg, speed, dur, denoise, pp, po, fmt, norm, progress=gr.Progress()):
                    opts = {"language": lang, "ref_text": ref_text, "num_step": steps,
                            "guidance_scale": cfg, "speed": speed, "duration": dur,
                            "denoise": denoise, "preprocess_prompt": pp, "postprocess_output": po,
                            "format": fmt, "normalize": True, "normalize_numbers": norm}
                    try:
                        return generate_voice(text, ref, json.dumps(opts), progress=progress)
                    except Exception as e:
                        _set_error(e)
                        return None, FAIL_PREFIX + type(e).__name__ + ": " + str(e)

                vc_btn.click(_vc_fn,
                             inputs=[vc_text, vc_ref, vc_lang, vc_ref_text, vc_steps, vc_cfg, vc_speed,
                                     vc_dur, vc_denoise, vc_pprompt, vc_pout, vc_fmt, vc_norm],
                             outputs=[vc_out, vc_status], api_name="generate_voice")

            with gr.Tab("Voice Design"):
                with gr.Row():
                    with gr.Column():
                        vd_text = gr.Textbox(label="Naskah Teks", lines=6, placeholder="Ketik naskah di sini...")
                        vd_instruct = gr.Textbox(
                            label="Deskripsi Suara (instruct)", lines=2,
                            placeholder="contoh: male, calm tone, moderate pitch",
                            info="Diisi aplikasi web otomatis. Bisa juga dipakai manual di sini.")
                        vd_lang = gr.Dropdown(LANGUAGES, value=DEFAULT_LANGUAGE, label="Bahasa")
                        vd_cats = []
                        with gr.Row():
                            for name, choices in CATEGORIES.items():
                                vd_cats.append(gr.Dropdown(choices, value="Otomatis", label=name))
                        with gr.Accordion("Pengaturan Lanjutan", open=False):
                            vd_steps = gr.Slider(8, 64, value=32, step=1, label="Inference Steps")
                            vd_cfg = gr.Slider(0.0, 4.0, value=2.0, step=0.1, label="Guidance Scale (CFG)")
                            vd_speed = gr.Slider(0.5, 1.5, value=1.0, step=0.05, label="Kecepatan")
                            vd_dur = gr.Number(label="Durasi Detik (0 = otomatis)", value=0)
                            vd_fmt = gr.Dropdown(["wav", "mp3"], value="wav", label="Format")
                            vd_norm = gr.Checkbox(label="Normalisasi Angka", value=True)
                    with gr.Column():
                        vd_out = gr.Audio(label="Hasil Audio", type="filepath")
                        vd_status = gr.Textbox(label="Status", value="🟢 Siap meracik suara.", lines=3)
                        vd_btn = gr.Button("Generate Voice Design", variant="primary")

                def _vd_fn(text, instruct, lang, steps, cfg, speed, dur, fmt, norm, *groups):
                    style = (instruct or "").strip() or _instruct_from(groups)
                    opts = {"language": lang, "num_step": steps, "guidance_scale": cfg,
                            "speed": speed, "duration": dur, "format": fmt, "normalize": True,
                            "normalize_numbers": norm}
                    try:
                        return voice_design(text, style, json.dumps(opts))
                    except Exception as e:
                        _set_error(e)
                        return None, FAIL_PREFIX + type(e).__name__ + ": " + str(e)

                vd_btn.click(_vd_fn,
                             inputs=[vd_text, vd_instruct, vd_lang, vd_steps, vd_cfg, vd_speed, vd_dur,
                                     vd_fmt, vd_norm, *vd_cats],
                             outputs=[vd_out, vd_status], api_name="voice_design")
    return demo

demo = build_demo()
print("Antarmuka siap. Endpoint:", ["generate_voice", "voice_design"])

## 📡 6. Jalankan server + buka tunnel publik

In [ ]:
CODE = FIXED_CODE if re.fullmatch(r"\d{6}", str(FIXED_CODE or "")) else f"{random.randint(0, 999999):06d}"
PUBLIC_URL = ""
quiet = io.StringIO()
try:
    with redirect_stdout(quiet), redirect_stderr(quiet):
        from pycloudflared import try_cloudflare
        tunnel = try_cloudflare(port=PORT)
    for attr in ("tunnel", "tunnel_url", "url"):
        val = getattr(tunnel, attr, None)
        if val:
            PUBLIC_URL = str(val)
            break
    if not PUBLIC_URL:
        m = re.search(r"https://[\w.-]+\.trycloudflare\.com", str(tunnel))
        PUBLIC_URL = m.group(0) if m else ""
except Exception as e:
    print("[Tunnel] pycloudflared gagal:", e)

USE_SHARE = not bool(PUBLIC_URL)
print("Tunnel Cloudflare:", PUBLIC_URL or "(gagal)")

app, LOCAL_URL, SHARE_URL = demo.queue().launch(
    server_name="0.0.0.0", server_port=PORT,
    share=USE_SHARE, inline=False, quiet=True, prevent_thread_lock=True, show_error=True,
)
if USE_SHARE:
    PUBLIC_URL = (SHARE_URL or "").strip()
print("Server lokal:", LOCAL_URL)

if PUBLIC_URL:
    ok = False
    for i in range(6):
        try:
            r = requests.get(PUBLIC_URL.rstrip("/") + "/gradio_api/info", timeout=20)
            if r.status_code == 200:
                ok = True
                break
        except Exception:
            pass
        time.sleep(5)
    print("Uji URL publik:", "OK" if ok else "BELUM MERESPONS")
else:
    print("[Tunnel] TIDAK ADA URL PUBLIK")

## 🔑 7. Kode akses

In [ ]:
def show_banner(code, url):
    rows = [
        "=" * 52,
        "  VONIGEN BY ANIKI  ·  Engine Notebook",
        "-" * 52,
        f"  KODE AKSES ANDA : {code}",
        "  Masukkan 6 digit ini ke halaman login aplikasi.",
        "-" * 52,
        f"  Engine : {ENGINE_USED}",
        f"  GPU    : {GPU_NAME}",
        f"  URL    : {url}",
        "=" * 52,
    ]
    print("\n" + "\n".join(rows) + "\n")

show_banner(CODE, PUBLIC_URL)

## 🔗 8. Sambungkan ke aplikasi (heartbeat)

In [ ]:
NOTEBOOK_ID = str(uuid.uuid4())
STOP = False

def heartbeat_payload():
    note = ""
    if DEVICE == "cpu":
        note = "Runtime berjalan di CPU — akan sangat lambat. Ubah ke T4 GPU."
    return {
        "notebook_id": NOTEBOOK_ID,
        "secret": PAIRING_SECRET,
        "code": CODE,
        "url": PUBLIC_URL,
        "engine": ENGINE_USED,
        "gpu": GPU_NAME,
        "version": str(gr.__version__),
        "endpoint": "generate_voice",
        "api_base": "/gradio_api",
        "note": note,
        "last_error": LAST_ERROR,
        "jobs": JOB_COUNT,
    }

def heartbeat_loop():
    base = APP_URL.rstrip("/")
    fails = 0
    while not STOP:
        try:
            r = requests.post(base + "/api/notebook/heartbeat", json=heartbeat_payload(), timeout=25)
            if r.status_code == 200:
                data = r.json()
                if data.get("stop"):
                    print("Server meminta notebook berhenti:", data.get("message", ""))
                    return
                if not data.get("ok"):
                    print("Server menolak heartbeat:", str(data)[:300])
                elif fails:
                    print("Heartbeat terhubung kembali.")
                fails = 0
            else:
                fails += 1
                if fails in (1, 3, 10):
                    print(f"Heartbeat gagal (HTTP {r.status_code}): {r.text[:250]}")
        except Exception as e:
            fails += 1
            if fails in (1, 3, 10):
                print("Heartbeat gagal:", e)
        time.sleep(30)

threading.Thread(target=heartbeat_loop, daemon=True).start()
time.sleep(3)
print("Heartbeat aktif · sesi", NOTEBOOK_ID[:8], "· kode", CODE)
print("Buka aplikasi:", APP_URL)

## 🧷 9. Keep-alive

Biarkan sel ini tetap berjalan.

In [ ]:
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    STOP = True
    print("⛔ Dihentikan oleh pengguna.")